In [ ]:
println("Loading imports")
include("../src/naml.jl")

Consider the following real-valued, parametrised function
\begin{align*}
f_a: \mathbb{Q}_2 & \rightarrow \mathbb{R} \cup \{\infty\}\\
x & \mapsto |(x-a)(x-2a)(x-4a)|_2
\end{align*}
where $\mathbb{Q}_2$ are the 2-adic numbers, $|\cdot|_2$ is the 2-adic norm, $a$ is the parameter, and $x$ is the variable.


Fix data points $X = \{1,2,4\} \subset \mathbb{Q}_2$ and consider the following loss function:
\begin{align*}
\ell: \mathbb{Q}_2 & \rightarrow \mathbb{R} \cup \{\infty\}\\
a & \mapsto \sum_{x \in X} f_a(x)
\end{align*}

Clearly, $a=1$ minimizes $\ell$ with $\ell(1)=0$.  In the Berkovich
analytification of $\mathbb{Q}_2$, can you trace a path from $a=8$ to $a=1$ whilst
only going in a direction with negative directional derivative of $\ell$?

In [ ]:
println("Initialising data")
prec = 20
K = PadicField(2, prec)

# Set up the 3 data points in the 2-adic numbers
c1 = [K(1)]
r1 = Vector{Int}([prec])
c2 = [K(2)]
r2 = Vector{Int}([prec])
c3 = [K(4)]
r3 = Vector{Int}([prec])
p1 = ValuationPolydisc(c1, r1) 
p2 = ValuationPolydisc(c2, r2) 
p3 = ValuationPolydisc(c3, r2)

# We're trying to fit the data points (x, y) = {(1, 0), (2, 0), (4, 0)}
# using the function g_a(x) = (x-a)(x-2a)(x-4a)
data = [(p1, 0), (p2, 0), (p3, 0)]
R, (x, a) =  polynomial_ring(K, ["x", "a"])

# Initialise g_a(x) using a wrapper for absolute polynomials and their sums
g = AbsolutePolynomialSum([(x-a)*(x-2*a)*(x-4*a)])
# Specify that x is the data variable and a is the parameter variable
f = AbstractModel(g, [true, false])

In [ ]:
println("Initialising optimisation tools")

next_branch = 1
config = (false, 1)
initial = ValuationPolydisc([K(1)], [0])
# Let's pick the Gauss point as our initial parameter
# ℓ is the loss function of this model wrt the L¹ norm 
ℓ = MPE_loss_init(f, data, 1)
# We optimise using Greedy descent
greedy_optim::OptimSetup = greedy_descent_init(initial, ℓ, next_branch, config)

In [ ]:
println("Optimising parameter")
# Compute the loss 

println("Loss before starting greedy descent is ", eval_loss(greedy_optim))

# Make 20 steps using greedy descent
N_epochs = 20
t1 = time()
for i in 1:N_epochs
    println("Loss at epoch ", i, " is ", eval_loss(greedy_optim))
    step!(greedy_optim)
end 
t2 = time()

# Compute the new value of the loss
println("Greedy descent finished in ", t2-t1, " seconds. \nThe Final parameter a is ", greedy_optim.param)

In [ ]:
gradient_state = 1
gradient_config = 1


# Declare new model with same starting point as the first one
new_model = Model(f, ValuationPolydisc([K(1)], [0]))
# Now let's optimise using gradient descent 
gradient_optim::OptimSetup  = gradient_descent_init(initial, ℓ, gradient_config, gradient_state)

println("Loss before starting gradient descent is ", eval_loss(gradient_optim))

t3 = time()
for i in 1:N_epochs
    println("Loss at epoch ", i, " is ", eval_loss(gradient_optim))
    step!(gradient_optim)
end 
t4 = time()

# Compute the new value of the loss
println("Gradient descent finished in ", t4-t3, " Seconds. \nThe Final parameter a is ", gradient_optim.param)
